# 00. Feature Table 생성 (00_feature_table.ipynb)

## 개요
이 노트북은 ML 학습에 사용할 피처 테이블을 생성합니다.  
**모든 ML 노트북(`01` ~ `04`)은 이 노트북이 생성한 CSV를 기준으로 실행됩니다.**

---

## 버전별 포함 피처

| 버전 | 추가 피처 | 출력 파일 |
|------|-----------|-----------|
| **v1** | 기본 피처 (메타 + 시간 + Star/Brand Power + 경쟁환경) | `feature_table_v1.csv` |
| **v2** | v1 + 실제 관람 요일 비율 (`weekend_ratio`, `weekday_0~4_ratio`) | `feature_table_v2.csv` |
| **v3** | v2 + 네이버 검색 트렌드 (`trend_pre7_avg` 등) | `feature_table_v3.csv` |

## 버전 관리 규칙
> ⚠️ **v1은 수정 금지** — 모든 모델의 성능 기준점(베이스라인)  
> 새 피처 추가 시 셀을 아래에 추가하고 `셀 0`의 `VERSION`만 올릴 것  
> 기존 컬럼명 변경 금지 — 새 피처 추가만 허용  
> 노트북 동시 편집 금지 — 커밋/푸시 후 다음 사람 pull

## 실행 방법
```
v1 생성: 셀 0 → VERSION='v1' 설정 후 Run All
v2 생성: 셀 0 → VERSION='v2' 설정 후 Run All
v3 생성: 셀 0 → VERSION='v3' 설정 후 Run All
```
→ VERSION에 따라 포함할 피처 셀이 자동으로 결정됩니다.

---

## 타겟 변수 3개 (고정)

| 컬럼 | 설명 | 용도 |
|------|------|------|
| `total_audience` | `MAX(audiAcc)` — 최종 누적 관객수 | 회귀 타겟 원본 |
| `log_audience` | `log1p(total_audience)` | 회귀 타겟 (실제 학습용, 왜도 완화) |
| `hit_class` | 관객수 구간 0~3 | 분류 타겟 |

```
hit_class 기준 (셀 0의 HIT_BINS에서 수정 가능):
  0: ~100만       (소규모)
  1: 100만~300만  (중규모)
  2: 300만~500만  (흥행)
  3: 500만~       (대흥행)
```

---

## ⚠️ 데이터 누수 원칙
**개봉 전에 알 수 있는 정보만 피처로 사용합니다.**

| ✅ 사용 가능 (개봉 전) | ❌ 사용 금지 (개봉 후) |
|----------------------|---------------------|
| genre, rating, nation, runtime, open_date | rank, audiCnt, salesAmt, scrnCnt |
| 감독/배우/배급사/제작사의 **과거작** 실적 | 해당 영화 자신의 box office 데이터 |
| 개봉일 기준 시즌/공휴일 정보 | — |
| 개봉 **전** 예매 요일 패턴 (v2) | 개봉 **후** 실관람 요일 패턴 |
| 개봉 **전** 네이버 검색 트렌드 (v3) | 개봉 **후** 검색량 |

---

## 전체 흐름
```
[셀 0]   환경 설정 및 VERSION 지정
[셀 1]   DB 전체 로드 (쿼리 한 곳에서 관리)
[셀 2]   타겟 변수 생성  ← v1 시작
[셀 3]   영화 메타 피처
[셀 4]   시간/계절 피처
[셀 5]   Star Power 피처
[셀 6]   Brand Power 피처
[셀 7]   경쟁 환경 피처
[셀 8]   이상치 제거
[셀 9]   v1 저장  ← v1 끝
[셀 10]  [v2] 실제 관람 요일 비율 피처
[셀 11]  v2 저장  ← v2 끝
[셀 12]  [v3] 네이버 검색 트렌드 피처
[셀 13]  v3 저장  ← v3 끝
```

---
## ⚙️ 셀 0. 환경 설정

**여기서 VERSION만 바꾸고 Run All 하면 됩니다.**

| 설정값 | 설명 |
|--------|------|
| `VERSION` | 생성할 CSV 버전. `'v1'` / `'v2'` / `'v3'` 중 선택 |
| `CUTOFF_DATE` | 이 날짜 이전 개봉작만 수집 |
| `HIT_BINS` | hit_class 구간 기준 (관객수, 단위: 명) |
| `STAR_POWER_TOP_N` | 주연 배우 Star Power 계산에 포함할 상위 배우 수 |

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd

from data.db import db

warnings.filterwarnings('ignore')

# ════════════════════════════════════════════════
# ✏️  여기만 수정하세요
VERSION          = 'v4'          # 'v1' / 'v2' / 'v3'
CUTOFF_DATE      = '2025-11-01'  # 이 날짜 이전 개봉작만 사용
STAR_POWER_TOP_N = 3             # 주연 배우 Star Power 상위 N명

HIT_BINS = [
    0,
    3_000_000,
    float('inf')
]

HIT_LABELS = [0, 1] # 일반 : 0, 흥행 : 1
# ════════════════════════════════════════════════

# VERSION별 포함 피처 정의
# 각 버전은 이전 버전의 피처를 모두 포함합니다 (누적)
VERSION_FEATURES = {
    'v1': ['meta', 'time', 'star', 'brand', 'market'],
    'v2': ['meta', 'time', 'star', 'brand', 'market', 'weekday'],
    'v3': ['meta', 'time', 'star', 'brand', 'market', 'weekday', 'trend'],
    'v4': ['meta', 'time', 'star', 'brand', 'market', 'trend']
}
ACTIVE_FEATURES = VERSION_FEATURES.get(VERSION, VERSION_FEATURES['v1'])

OUTPUT_DIR  = 'data/processed'
OUTPUT_PATH = f'{OUTPUT_DIR}/feature_table_{VERSION}.csv'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# hit_class 설정을 JSON으로 저장 (ML 노트북에서 참조용)
with open('hit_config.json', 'w', encoding='utf-8') as f:
    json.dump({'HIT_BINS': HIT_BINS, 'HIT_LABELS': HIT_LABELS}, f, ensure_ascii=False)

print(f'✅ 설정 완료')
print(f'   VERSION       : {VERSION}')
print(f'   출력 경로     : {OUTPUT_PATH}')
print(f'   포함 피처 그룹: {ACTIVE_FEATURES}')
print(f'   hit_class 기준: {HIT_BINS}')

✅ 설정 완료
   VERSION       : v4
   출력 경로     : data/processed/feature_table_v4.csv
   포함 피처 그룹: ['meta', 'time', 'star', 'brand', 'market', 'trend']
   hit_class 기준: [0, 3000000, inf]


---
## 🗄️ 셀 1. DB 전체 로드

모든 쿼리를 한 곳에서 관리합니다.  
피처 셀에서는 이미 로드된 변수를 DataFrame으로 변환하여 사용합니다.

| 변수 | 용도 |
|------|------|
| `boxoffice_rows` | 타겟 변수 + 요일 피처 |
| `movie_rows` | 영화 메타 피처 |
| `holiday_rows` | 시간/계절 피처 |
| `casting_rows` | Star Power 피처 |
| `company_rows` | Brand Power 피처 |
| `new_rows` | 경쟁 환경 피처 (신작 개봉일) |
| `market_rows` | 경쟁 환경 피처 (시장 통계) |
| `trend_rows` | 네이버 트렌드 피처 (v3 전용) |

In [21]:
print('🔄 DB 로드 시작...')

# 1. 박스오피스 (타겟 + 요일 피처용)
boxoffice_rows = db.fetch_all("""
    SELECT movie_id, target_date, audiCnt, audiAcc, salesAcc
    FROM daily_box_office
""")
print(f'  ✅ boxoffice_rows: {len(boxoffice_rows):,}건')

# 2. 영화 메타
movie_rows = db.fetch_all("""
    SELECT movie_id, title, genre, rating, nation, open_date, runtime
    FROM movies
    WHERE open_date < %s
""", (CUTOFF_DATE,))
print(f'  ✅ movie_rows    : {len(movie_rows):,}건')

# 3. 공휴일
holiday_rows = db.fetch_all('SELECT holiday_date, is_weekend_effect FROM holidays')
print(f'  ✅ holiday_rows  : {len(holiday_rows):,}건')

# 4. 캐스팅 (Star Power)
casting_rows = db.fetch_all("""
    SELECT mc.movie_id, mc.person_id, mc.cast_role, p.name
    FROM movie_casting mc
    JOIN people p ON mc.person_id = p.person_id
""")
print(f'  ✅ casting_rows  : {len(casting_rows):,}건')

# 5. 영화사 (Brand Power)
company_rows = db.fetch_all("""
    SELECT cp.movie_id, cp.company_id, cp.part_role, c.company_name
    FROM company_part cp
    JOIN companys c ON cp.company_id = c.company_id
    WHERE cp.part_role IN ('배급사', '제작사')
""")
print(f'  ✅ company_rows  : {len(company_rows):,}건')

# 6. 신작 개봉일 (경쟁 환경)
new_rows = db.fetch_all("""
    SELECT movie_id, MIN(target_date) AS first_date
    FROM daily_box_office
    WHERE rankOldAndNew = 'NEW'
    GROUP BY movie_id
""")
print(f'  ✅ new_rows      : {len(new_rows):,}건')

# 7. 시장 통계 (경쟁 환경)
market_rows = db.fetch_all("""
    SELECT target_date, total_audi
    FROM daily_market_stats
    ORDER BY target_date
""")
print(f'  ✅ market_rows   : {len(market_rows):,}건')

# 8. 네이버 트렌드 (v3 전용 — 테이블 없으면 빈 리스트)
trend_rows = []
if 'trend' in ACTIVE_FEATURES:
    try:
        trend_rows = db.fetch_all("""
            SELECT movie_id, trend_date, search_index
            FROM naver_search_trend
            ORDER BY movie_id, trend_date
        """)
        print(f'  ✅ trend_rows    : {len(trend_rows):,}건')
    except Exception:
        print(f'  ⚠️  trend_rows    : 테이블 없음 → 0으로 채움')

print(f'\n✅ DB 로드 완료')

🔄 DB 로드 시작...
  ✅ boxoffice_rows: 58,440건
  ✅ movie_rows    : 3,908건
  ✅ holiday_rows  : 204건
  ✅ casting_rows  : 50,575건
  ✅ company_rows  : 5,439건
  ✅ new_rows      : 2,331건
  ✅ market_rows   : 5,844건
  ✅ trend_rows    : 83,575건

✅ DB 로드 완료


---
## 🎯 셀 2. 타겟 변수 생성 (from `daily_box_office`)

```
MAX(audiAcc) 를 사용하는 이유:
  audiAcc = 누적값 → 날짜가 지날수록 단조 증가
  SUM(audiAcc) → 누적값을 또 더하므로 완전히 틀린 값 ❌
  MAX(audiAcc) → 마지막 날의 누적값 = 최종 총 관객수  ✅

log_audience 를 사용하는 이유:
  관객수는 right-skewed 분포 → 극단값에 모델이 끌려감
  log1p 적용 시 분포가 정규분포에 가까워짐
  복원: expm1(예측값) → 실제 관객수
```

In [22]:
daily_df = pd.DataFrame(boxoffice_rows)
daily_df['target_date'] = pd.to_datetime(daily_df['target_date'])
daily_df['audiCnt']     = pd.to_numeric(daily_df['audiCnt'],  errors='coerce')
daily_df['audiAcc']     = pd.to_numeric(daily_df['audiAcc'],  errors='coerce')
daily_df['salesAcc']    = pd.to_numeric(daily_df['salesAcc'], errors='coerce')

# 영화별 최종 누적 관객수/매출 (MAX = 마지막 날의 누적값)
df = daily_df.groupby('movie_id').agg(
    total_audience=('audiAcc',  'max'),
    total_sales   =('salesAcc', 'max'),
).reset_index()

# 타겟 변수 생성
df['log_audience'] = np.log1p(df['total_audience'])
df['hit_class']    = pd.cut(
    df['total_audience'],
    bins=HIT_BINS, labels=HIT_LABELS, right=False
).astype('Int64')

print(f'✅ 타겟 변수 생성: {len(df):,}편')
print(f'\nhit_class 분포:')
label_map = {0:'0(~100만)', 1:'1(100만~300만)', 2:'2(300만~500만)', 3:'3(500만~)'}
print(df['hit_class'].value_counts().sort_index().rename(label_map).to_string())

✅ 타겟 변수 생성: 3,943편

hit_class 분포:
hit_class
0(~100만)        3717
1(100만~300만)     226


---
## 🎬 셀 3. 영화 메타 피처 (from `movies`)

개봉 전에 확정되는 정보이므로 모두 사용 가능합니다.

| 피처 | 원본 컬럼 | 변환 방법 |
|------|----------|-----------|
| `genre` | `genre` | 문자열 카테고리 (ML 노트북에서 인코딩) |
| `rating_encoded` | `rating` | 순서형 인코딩 (전체=0 ≤ 12세=1 ≤ 15세=2 ≤ 청불=3) |
| `is_korean` | `nation` | 이진 (한국=1, 외국=0) |
| `runtime` | `runtime` | 수치형 그대로 (분 단위) |

> genre를 문자열로 저장하는 이유: 트리 모델은 LabelEncoding으로 충분하고,  
> 선형 모델은 OHE가 필요하므로 원본을 저장하고 각 ML 노트북에서 인코딩합니다.

In [23]:
df_movies = pd.DataFrame(movie_rows)
df_movies['open_date'] = pd.to_datetime(df_movies['open_date'], errors='coerce')
df_movies['runtime']   = pd.to_numeric(df_movies['runtime'],   errors='coerce')

# ── 결측치 / 센티널 값 처리 ──
# '미정'은 수집 단계의 기본값 → 실제 결측으로 변환
df_movies['genre']  = df_movies['genre'].replace('미정', '기타').fillna('기타')
df_movies['rating'] = df_movies['rating'].replace('미정', np.nan)
df_movies['nation'] = df_movies['nation'].replace('미정', np.nan)
# open_date 1900-01-01 = 개봉일 미상 → NaT 처리
df_movies.loc[df_movies['open_date'].dt.year == 1900, 'open_date'] = pd.NaT
# runtime 0 = 미상 → 중앙값으로 대체
df_movies.loc[df_movies['runtime'] == 0, 'runtime'] = np.nan
df_movies['runtime'] = df_movies['runtime'].fillna(df_movies['runtime'].median())

# ── rating 순서형 인코딩 ──
# 관람 가능한 연령이 낮을수록 잠재 관객 많음 → 낮은 숫자 = 제한 적음
RATING_ORDER = {
    '전체관람가': 0, '연소자관람가': 0, '모든 관람객이 관람할 수 있는 등급': 0, '전체': 0,
    '12세이상관람가': 1, '중학생이상관람가': 1, '12세관람가': 1, '12세': 1, '미정': 1,
    '15세이상관람가': 2, '고등학생이상관람가': 2, '15세관람가': 2,
    '15세 미만인 자는 관람할 수 없는 등급': 2, '15세': 2,
    '청소년관람불가': 3, '연소자관람불가': 3, '18세 미만인 자는 관람할 수 없는 등급': 3,
    '18세관람가': 3, '미성년자관람불가': 3,
}
df_movies['rating_encoded'] = df_movies['rating'].map(RATING_ORDER).fillna(2).astype(int)

# ── 국가: 한국 여부 이진화 ──
df_movies['is_korean'] = (df_movies['nation'] == '한국').astype(int)

meta_cols = ['movie_id', 'title', 'open_date', 'runtime', 'rating_encoded', 'is_korean', 'genre']
df = df.merge(df_movies[meta_cols], on='movie_id', how='left')

print(f'✅ 영화 메타 피처 병합 완료')
print(f'   genre 종류    : {df_movies["genre"].nunique()}개')
print(f'   rating 분포   :\n{df_movies["rating"].value_counts().to_string()}')

✅ 영화 메타 피처 병합 완료
   genre 종류    : 21개
   rating 분포   :
rating
15세이상관람가                   1275
전체관람가                      1016
12세이상관람가                   1015
청소년관람불가                     508
15세관람가                       14
12세관람가                       13
고등학생이상관람가                     9
연소자관람불가                       7
18세관람가                        7
중학생이상관람가                      6
연소자관람가                        6
15세 미만인 자는 관람할 수 없는 등급        4
18세 미만인 자는 관람할 수 없는 등급        2
                              1
모든 관람객이 관람할 수 있는 등급           1


---
## 📅 셀 4. 시간/계절 피처 (from `movies.open_date` + `holidays`)

| 피처 | 산출 방법 | 의미 |
|------|-----------|------|
| `open_date` | 개봉일 | movies의 'open_date' , type:date |
| `open_month` | 개봉월 | 여름(7~8)/겨울(12~1) 성수기 효과 |
| `open_day_of_week` | 개봉 요일 (0=월~6=일) | 수/목 개봉 vs 주말 개봉 |
| `is_summer` | 7~8월 여부 | 여름 성수기 이진 변수 |
| `is_winter` | 12~1월 여부 | 겨울 성수기 이진 변수 |
| `is_holiday_release` | 공휴일 당일 개봉 여부 | 공휴일 특수 |
| `holiday_nearby_count` | 개봉일 ±7일 내 공휴일 수 | 개봉 첫 주 연휴 효과 |

In [24]:
# ── 날짜 기반 피처 ──
df['open_month']       = df['open_date'].dt.month
df['open_day_of_week'] = df['open_date'].dt.dayofweek
df['is_summer']        = df['open_month'].isin([7, 8]).astype(int)
df['is_winter']        = df['open_month'].isin([12, 1]).astype(int)

# ── 공휴일 피처 ──
if holiday_rows:
    df_holidays = pd.DataFrame(holiday_rows)
    df_holidays['holiday_date'] = pd.to_datetime(df_holidays['holiday_date'])
    holiday_np = df_holidays['holiday_date'].values.astype('datetime64[D]')

    def get_holiday_features(open_dt, holiday_np, window=7):
        """개봉일 기준 ±window일 이내 공휴일 수와 공휴일 당일 개봉 여부를 반환합니다."""
        if pd.isna(open_dt):
            return 0, 0
        open_np   = np.datetime64(open_dt, 'D')
        diff_days = np.abs((holiday_np - open_np).astype(int))
        return int((diff_days == 0).any()), int((diff_days <= window).sum())

    results = df['open_date'].apply(
        lambda d: pd.Series(get_holiday_features(d, holiday_np),
                            index=['is_holiday_release', 'holiday_nearby_count'])
    )
    df = pd.concat([df, results], axis=1)
    print(f'✅ 공휴일 피처 생성 완료')
    print(f'   공휴일 당일 개봉 : {df["is_holiday_release"].sum()}편')
    print(f'   ±7일 내 공휴일 평균: {df["holiday_nearby_count"].mean():.2f}일')
else:
    df['is_holiday_release']   = 0
    df['holiday_nearby_count'] = 0
    print('⚠️  holidays 테이블 비어있음 → 공휴일 피처 0으로 설정')

print(f'✅ 시간/계절 피처 생성 완료 (여름: {df["is_summer"].sum()}편 / 겨울: {df["is_winter"].sum()}편)')

✅ 공휴일 피처 생성 완료
   공휴일 당일 개봉 : 188편
   ±7일 내 공휴일 평균: 0.50일
✅ 시간/계절 피처 생성 완료 (여름: 613편 / 겨울: 595편)


---
## 🌟 셀 5. Star Power 피처 (from `people` + `movie_casting` + `daily_box_office`)

**핵심 원칙: 해당 영화 개봉일 이전의 과거 실적만 참조 (데이터 누수 방지)**

| 피처 | 의미 |
|------|------|
| `director_avg_audi` | 감독의 과거 영화 평균 관객수 |
| `director_movie_count` | 감독의 과거 영화 편수 |
| `lead_actor_avg_audi` | 주연 배우 상위 N명의 평균 관객수 |
| `lead_actor_movie_count` | 주연 배우 평균 과거 편수 |
| `cast_max_star_power` | 출연진 중 최고 평균 관객수 (스타 1명 효과 포착) |

> 신인 감독/배우(과거 데이터 없음) → 0 (모델이 '검증된 실적 없음'으로 해석)

In [25]:
# ── 5-0. 공통 맵 준비 ──
open_date_map = df.set_index('movie_id')['open_date'].to_dict()
audi_map      = df.set_index('movie_id')['total_audience'].to_dict()
df_casting    = pd.DataFrame(casting_rows)

def calc_star_power(person_id, target_movie_id, df_casting, audi_map, open_date_map):
    """
    특정 영화인의 Star Power를 계산합니다.
    target_movie_id의 개봉일 이전에 개봉한 과거작의 평균 관객수를 반환합니다.
    Returns: (avg_audi, movie_count)
    """
    target_open = open_date_map.get(target_movie_id)
    if pd.isna(target_open):
        return 0.0, 0
    past_audis = [
        float(audi_map[mid])
        for mid in df_casting[df_casting['person_id'] == person_id]['movie_id']
        if mid != target_movie_id
        and not pd.isna(open_date_map.get(mid))
        and open_date_map[mid] < target_open
        and mid in audi_map
        and not np.isnan(float(audi_map[mid]))
    ]
    return (float(np.mean(past_audis)), len(past_audis)) if past_audis else (0.0, 0)

# ── 5-1. 감독 Star Power ──
df_directors   = df_casting[df_casting['cast_role'] == '감독'].drop_duplicates('movie_id', keep='first')
director_stats = [
    {'movie_id': r['movie_id'],
     **dict(zip(['director_avg_audi', 'director_movie_count'],
                calc_star_power(r['person_id'], r['movie_id'], df_casting, audi_map, open_date_map)))}
    for _, r in df_directors.iterrows()
]
df = df.merge(pd.DataFrame(director_stats), on='movie_id', how='left')
df['director_avg_audi']    = df['director_avg_audi'].fillna(0)
df['director_movie_count'] = df['director_movie_count'].fillna(0).astype(int)
print('✅ 감독 Star Power 계산 완료')

# ── 5-2. 배우 Star Power ──
df_actors    = df_casting[df_casting['cast_role'] == '배우']
actor_stats  = []
for movie_id, group in df_actors.groupby('movie_id'):
    sp_list = [calc_star_power(pid, movie_id, df_casting, audi_map, open_date_map)
               for pid in group['person_id']]
    sp_sorted = sorted(sp_list, key=lambda x: x[0], reverse=True)
    top_n     = sp_sorted[:STAR_POWER_TOP_N]
    top_audis = [s[0] for s in top_n if s[0] > 0]
    actor_stats.append({
        'movie_id'              : movie_id,
        'lead_actor_avg_audi'   : float(np.mean(top_audis)) if top_audis else 0.0,
        'lead_actor_movie_count': int(np.mean([s[1] for s in top_n])) if top_n else 0,
        'cast_max_star_power'   : float(sp_sorted[0][0]) if sp_sorted else 0.0,
    })
df = df.merge(pd.DataFrame(actor_stats), on='movie_id', how='left')
for col in ['lead_actor_avg_audi', 'cast_max_star_power']:
    df[col] = df[col].fillna(0)
df['lead_actor_movie_count'] = df['lead_actor_movie_count'].fillna(0).astype(int)
print('✅ 배우 Star Power 계산 완료')

✅ 감독 Star Power 계산 완료
✅ 배우 Star Power 계산 완료


---
## 🏢 셀 6. Brand Power 피처 (from `companys` + `company_part` + `daily_box_office`)

Star Power와 동일한 원칙으로 **개봉일 이전 과거작만** 참조합니다.  
배급사(마케팅/배급망 역량)와 제작사(제작 품질 역량)를 분리하여 피처를 만듭니다.

| 피처 | 의미 |
|------|------|
| `distributor_avg_audi` | 배급사의 과거 영화 평균 관객수 |
| `distributor_movie_count` | 배급사의 과거 영화 편수 |
| `producer_avg_audi` | 제작사의 과거 영화 평균 관객수 |
| `producer_movie_count` | 제작사의 과거 영화 편수 |

In [26]:
df_company = pd.DataFrame(company_rows)

def calc_company_power(company_id, target_movie_id, df_company, audi_map, open_date_map):
    """특정 영화사의 Brand Power를 계산합니다. (개봉일 이전 과거작만 참조)"""
    target_open = open_date_map.get(target_movie_id)
    if pd.isna(target_open):
        return 0.0, 0
    past_audis = [
        float(audi_map[mid])
        for mid in df_company[df_company['company_id'] == company_id]['movie_id']
        if mid != target_movie_id
        and not pd.isna(open_date_map.get(mid))
        and open_date_map[mid] < target_open
        and mid in audi_map
        and not np.isnan(float(audi_map[mid]))
    ]
    return (float(np.mean(past_audis)), len(past_audis)) if past_audis else (0.0, 0)

for role, avg_col, cnt_col in [
    ('배급사', 'distributor_avg_audi', 'distributor_movie_count'),
    ('제작사', 'producer_avg_audi',    'producer_movie_count'),
]:
    df_role  = df_company[df_company['part_role'] == role].drop_duplicates('movie_id', keep='first')
    stats    = [
        {'movie_id': r['movie_id'],
         **dict(zip([avg_col, cnt_col],
                    calc_company_power(r['company_id'], r['movie_id'], df_company, audi_map, open_date_map)))}
        for _, r in df_role.iterrows()
    ]
    df = df.merge(pd.DataFrame(stats), on='movie_id', how='left')
    df[avg_col] = df[avg_col].fillna(0)
    df[cnt_col] = df[cnt_col].fillna(0).astype(int)
    print(f'✅ {role} Brand Power 계산 완료')

✅ 배급사 Brand Power 계산 완료
✅ 제작사 Brand Power 계산 완료


---
## ⚔️ 셀 7. 경쟁 환경 피처 (from `daily_box_office` + `daily_market_stats`)

| 피처 | 의미 |
|------|------|
| `same_week_releases` | 개봉일 ±3일 내 다른 신작 수 (경쟁 강도) |
| `market_avg_audi_7d` | 개봉 직전 7일 시장 평균 관객수 (시장 활성도) |

In [27]:
# ── 7-1. 동시 개봉 신작 수 ──
df_new    = pd.DataFrame(new_rows)
df_new['first_date'] = pd.to_datetime(df_new['first_date'])
new_dates = df_new['first_date'].values.astype('datetime64[D]')

def count_same_week_releases(open_dt, new_dates, window=3):
    """개봉일 ±window일 이내의 다른 신작 수를 반환합니다 (자기 자신 제외)."""
    if pd.isna(open_dt):
        return 0
    diff_days = np.abs((new_dates - np.datetime64(open_dt, 'D')).astype(int))
    return max(0, int((diff_days <= window).sum()) - 1)

df['same_week_releases'] = df['open_date'].apply(lambda d: count_same_week_releases(d, new_dates))
print('✅ 경쟁 환경 피처(same_week_releases) 생성 완료')

# ── 7-2. 개봉 직전 7일 시장 평균 관객수 ──
df_market = pd.DataFrame(market_rows)
df_market['target_date'] = pd.to_datetime(df_market['target_date'])
df_market['total_audi']  = pd.to_numeric(df_market['total_audi'], errors='coerce')
df_market = df_market.set_index('target_date').sort_index()

def get_market_avg_7d(open_dt, df_market):
    """개봉일 직전 7일간의 시장 평균 관객수를 반환합니다."""
    if pd.isna(open_dt):
        return np.nan
    window = df_market.loc[open_dt - pd.Timedelta(days=7) : open_dt - pd.Timedelta(days=1), 'total_audi']
    return float(window.mean()) if len(window) > 0 else np.nan

df['market_avg_audi_7d'] = df['open_date'].apply(lambda d: get_market_avg_7d(d, df_market))
df['market_avg_audi_7d'] = df['market_avg_audi_7d'].fillna(df['market_avg_audi_7d'].median())
print('✅ 시장 활성도 피처(market_avg_audi_7d) 생성 완료')

✅ 경쟁 환경 피처(same_week_releases) 생성 완료
✅ 시장 활성도 피처(market_avg_audi_7d) 생성 완료


---
## 🗑️ 셀 8. 이상치 제거

| 조건 | 이유 |
|------|------|
| `total_audience <= 0` | 실질적인 상영 기록 없음 |
| `open_date` NaT | 개봉일 미상 → 시간/계절 피처 전부 무의미 |

In [28]:
before = len(df)
df = df[df['total_audience'] > 0]
df = df[df['open_date'].notna()]
after  = len(df)
print(f'✅ 이상치 제거: {before:,}편 → {after:,}편 ({before - after}편 제거)')

null_summary = df.isna().sum()
null_summary = null_summary[null_summary > 0]
if len(null_summary) > 0:
    print(f'⚠️  잔여 결측치:\n{null_summary.to_string()}')
else:
    print('✅ 잔여 결측치 없음')
print(f'현재 데이터셋: {df.shape[0]:,}행 × {df.shape[1]}열')

✅ 이상치 제거: 3,943편 → 3,869편 (74편 제거)
✅ 잔여 결측치 없음
현재 데이터셋: 3,869행 × 28열


---
## 💾 셀 9. v1 저장

**v1 포함 피처: 메타 + 시간/계절 + Star Power + Brand Power + 경쟁환경**

```
VERSION = 'v1' 일 때만 저장합니다.
v2, v3는 아래 셀에서 피처를 추가한 뒤 각 버전 저장 셀에서 저장됩니다.
```

In [29]:
# ── v1 컬럼 정의 ──
ID_COLS     = ['movie_id', 'title']
TARGET_COLS = ['total_audience', 'log_audience', 'hit_class']

V1_FEATURE_COLS = (
    ['runtime', 'rating_encoded', 'is_korean', 'genre']            # 메타
  + ['open_date','open_month', 'open_day_of_week', 'is_summer', 'is_winter',
     'is_holiday_release', 'holiday_nearby_count']                 # 시간/계절
  + ['director_avg_audi', 'director_movie_count',
     'lead_actor_avg_audi', 'lead_actor_movie_count',
     'cast_max_star_power']                                        # Star Power
  + ['distributor_avg_audi', 'distributor_movie_count',
     'producer_avg_audi', 'producer_movie_count']                  # Brand Power
  + ['same_week_releases', 'market_avg_audi_7d']                   # 경쟁 환경
)

if VERSION == 'v1':
    _save_cols = ID_COLS + V1_FEATURE_COLS + TARGET_COLS
    _save_cols = [c for c in _save_cols if c in df.columns]
    df_out = df[_save_cols].copy()
    df_out['total_audience'] = df_out['total_audience'].astype(int)
    df_out['hit_class']      = df_out['hit_class'].astype(int)
    df_out.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
    print(f'✅ v1 저장 완료: {OUTPUT_PATH}')
    print(f'   shape: {df_out.shape[0]:,}편 × {df_out.shape[1]}컬럼')
    print(f'   컬럼 목록: {list(df_out.columns)}')
else:
    print(f'ℹ️  VERSION={VERSION} → v1 저장 건너뜀. 아래 셀에서 저장합니다.')

ℹ️  VERSION=v4 → v1 저장 건너뜀. 아래 셀에서 저장합니다.


---
## 📅 셀 10. [v2 추가] 실제 관람 요일 비율 피처

**v2부터 포함됩니다. VERSION이 v1이면 이 셀은 자동으로 건너뜁니다.**

개봉 전 예매 데이터 기반 요일별 관람 비율입니다.

**주의! 해당 feature는 개봉 후 결과적인 데이터이기 때문에 feature leakage에 해당합니다.**
- ML진행 시에는 해당 feature를 제거한 v4데이터를 사용해야 합니다.

| 피처 | 의미 |
|------|------|
| `weekend_ratio` | 주말(토+일) 관람 비율 |
| `weekday_0_ratio` ~ `weekday_4_ratio` | 월~금 요일별 관람 비율 (5개) |

> 제거한 컬럼: `weekday_ratio`(1-weekend_ratio와 동일), `sat_ratio`, `sun_ratio`(weekend_ratio에 포함),  
> `weekday_6_ratio`(나머지 6개 합으로 계산 가능 → 다중공선성)

In [30]:
if 'weekday' not in ACTIVE_FEATURES:
    print(f'ℹ️  VERSION={VERSION} → 요일 비율 피처 건너뜀')
else:
    daily_df['weekday'] = daily_df['target_date'].dt.dayofweek
    total_by_movie      = daily_df.groupby('movie_id')['audiCnt'].sum()

    # 주말 비율 (토=5, 일=6)
    weekend_ratio = (
        daily_df[daily_df['weekday'] >= 5]
        .groupby('movie_id')['audiCnt'].sum()
        / total_by_movie
    ).fillna(0)
    df['weekend_ratio'] = df['movie_id'].map(weekend_ratio)

    # 요일별 비율 (월=0 ~ 금=4, 토=5 제외, 일=6 제외 → 다중공선성 방지)
    for day in range(5):  # 0~4: 월~금
        day_ratio = (
            daily_df[daily_df['weekday'] == day]
            .groupby('movie_id')['audiCnt'].sum()
            / total_by_movie
        ).fillna(0)
        df[f'weekday_{day}_ratio'] = df['movie_id'].map(day_ratio)

    print(f'✅ 요일 비율 피처 생성 완료')
    print(f'   weekend_ratio 평균: {df["weekend_ratio"].mean():.3f}')

ℹ️  VERSION=v4 → 요일 비율 피처 건너뜀


---
## 💾 셀 11. v2 저장

**v2 포함 피처: v1 전체 + 요일 비율 피처**

In [31]:
if VERSION == 'v2':
    WEEKDAY_COLS = (
        ['weekend_ratio']
      + [f'weekday_{d}_ratio' for d in range(5)]  # 월~금 5개
    )
    _save_cols = ID_COLS + V1_FEATURE_COLS + WEEKDAY_COLS + TARGET_COLS
    _save_cols = [c for c in _save_cols if c in df.columns]
    df_out = df[_save_cols].copy()
    df_out['total_audience'] = df_out['total_audience'].astype(int)
    df_out['hit_class']      = df_out['hit_class'].astype(int)
    df_out.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
    print(f'✅ v2 저장 완료: {OUTPUT_PATH}')
    print(f'   shape: {df_out.shape[0]:,}편 × {df_out.shape[1]}컬럼')
    print(f'   v1 대비 추가 컬럼: {WEEKDAY_COLS}')
else:
    print(f'ℹ️  VERSION={VERSION} → v2 저장 건너뜀')

ℹ️  VERSION=v4 → v2 저장 건너뜀


---
## 🔍 셀 12. [v3 추가] 네이버 검색 트렌드 피처 (from `naver_search_trend`)

**v3부터 포함됩니다. VERSION이 v1/v2이면 이 셀은 자동으로 건너뜁니다.**

**개봉 전** 검색 트렌드만 사용합니다 (개봉 후 데이터 제외 → 누수 없음).

| 피처 | 의미 |
|------|------|
| `trend_pre7_avg` | 개봉일 -7일 ~ -1일 평균 검색량 (직전 관심도) |
| `trend_pre7_max` | 개봉 직전 7일 최대 검색량 (화제성 피크) |
| `trend_growth_rate` | pre7_avg / pre30_avg (임박 상승률) |
| `relative_search_share` | (내 영화 pre30_avg) / (개봉 직전 30일 내 개봉 Top 5 영화 pre30_avg 합) |
| `has_trend_data` | 트렌드 데이터 존재 여부 (2010~2016 등 과거 영화 구분) |

In [ ]:
TREND_COLS = ['trend_pre7_avg', 'trend_pre7_max',
              'trend_growth_rate', 'relative_search_share', 'has_trend_data']

if 'trend' not in ACTIVE_FEATURES:
    print(f'ℹ️  VERSION={VERSION} → 트렌드 피처 건너뜀')
elif not trend_rows:
    for col in TREND_COLS:
        df[col] = 0
    print('⚠️  naver_search_trend 테이블 비어있음 → 트렌드 피처 0으로 설정')
else:
    df_trend = pd.DataFrame(trend_rows)
    df_trend['trend_date']   = pd.to_datetime(df_trend['trend_date'])
    df_trend['search_index'] = pd.to_numeric(df_trend['search_index'], errors='coerce')

    def calc_trend_features(movie_id, df_trend, open_date_map):
        """
        개봉일 이전 검색 트렌드 피처를 계산합니다 (누수 방지).
        - trend_pre7_avg: 개봉 -7~-1일 평균
        - trend_growth_rate: pre7_avg / pre30_avg
        - relative_search_share: (내 pre30_avg) / (개봉 직전 30일 Top 5 영화들의 pre30_avg 합)
        """
        default  = dict(zip(TREND_COLS, [0.0, 0.0, 0.0, 0.0, 0]))
        open_dt  = open_date_map.get(movie_id)
        if pd.isna(open_dt):
            return default
        pre_data = df_trend[(df_trend['movie_id'] == movie_id) & (df_trend['trend_date'] < open_dt)]
        if pre_data.empty:
            return default

        # ── 이 영화의 검색량 계산 ──
        pre7  = pre_data[pre_data['trend_date'] >= open_dt - pd.Timedelta(days=7)]
        pre30 = pre_data[pre_data['trend_date'] >= open_dt - pd.Timedelta(days=30)]
        pre7_avg  = float(pre7['search_index'].mean())  if not pre7.empty  else 0.0
        pre30_avg = float(pre30['search_index'].mean()) if not pre30.empty else 0.0
        pre7_max  = float(pre7['search_index'].max())   if not pre7.empty  else 0.0

        # ── 경쟁 영화 선정: 개봉일 -30~-1일 내 개봉한 영화들 ──
        start_date = open_dt - pd.Timedelta(days=30)
        end_date   = open_dt - pd.Timedelta(days=1)
        competing_movies = [
            mid for mid, dt in open_date_map.items()
            if not pd.isna(dt) and start_date <= dt <= end_date
        ]

        # ── 경쟁 영화들의 pre30_avg 계산 ──
        competing_pre30_avgs = []
        for mid in competing_movies:
            mid_open = open_date_map[mid]
            mid_pre_data = df_trend[(df_trend['movie_id'] == mid) & (df_trend['trend_date'] < mid_open)]
            if not mid_pre_data.empty:
                mid_pre30 = mid_pre_data[mid_pre_data['trend_date'] >= mid_open - pd.Timedelta(days=30)]
                if not mid_pre30.empty:
                    competing_pre30_avgs.append(float(mid_pre30['search_index'].mean()))

        # ── Top 5 선정 및 relative_search_share 계산 ──
        top5_avgs = sorted(competing_pre30_avgs, reverse=True)[:5]
        top5_sum = sum(top5_avgs)
        relative_search_share = (pre30_avg / top5_sum) if top5_sum > 0 else 0.0

        return {
            'trend_pre7_avg'      : pre7_avg,
            'trend_pre7_max'      : pre7_max,
            'trend_growth_rate'   : (pre7_avg / pre30_avg) if pre30_avg > 0 else 0.0,
            'relative_search_share': relative_search_share,
            'has_trend_data'      : 1,
        }

    trend_stats = [{'movie_id': mid, **calc_trend_features(mid, df_trend, open_date_map)}
                   for mid in df['movie_id']]
    df = df.merge(pd.DataFrame(trend_stats), on='movie_id', how='left')
    df[TREND_COLS] = df[TREND_COLS].fillna(0)
    
    print(f'✅ 네이버 트렌드 피처 생성 완료')
    print(f'   trend_pre7_avg 평균: {df["trend_pre7_avg"].mean():.2f}')
    print(f'   relative_search_share 평균: {df["relative_search_share"].mean():.4f}')
    print(f'   트렌드 있는 영화: {int(df["has_trend_data"].sum())}편 / 전체 {len(df)}편')

✅ 네이버 트렌드 피처 생성 완료
   trend_pre7_avg 평균: 6.39
   relative_search_share 평균: 0.0530
   트렌드 있는 영화: 2420편 / 전체 3869편


---
## 💾 셀 13. v3 저장

**v3 포함 피처: v2 전체 + 네이버 트렌드 피처 (5개)**
- `trend_pre7_avg`, `trend_pre7_max`, `trend_growth_rate`, `relative_search_share`, `has_trend_data`
- `trend_pre30_avg` 제거 (내부 계산용으로만 사용)

In [33]:
if VERSION == 'v3':
    WEEKDAY_COLS = (['weekend_ratio'] + [f'weekday_{d}_ratio' for d in range(5)])
    _save_cols   = ID_COLS + V1_FEATURE_COLS + WEEKDAY_COLS + TREND_COLS + TARGET_COLS
    _save_cols   = [c for c in _save_cols if c in df.columns]
    df_out = df[_save_cols].copy()
    df_out['total_audience'] = df_out['total_audience'].astype(int)
    df_out['hit_class']      = df_out['hit_class'].astype(int)
    df_out.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
    print(f'✅ v3 저장 완료: {OUTPUT_PATH}')
    print(f'   shape: {df_out.shape[0]:,}편 × {df_out.shape[1]}컬럼')
    print(f'   v2 대비 추가 컬럼: {TREND_COLS}')
else:
    print(f'ℹ️  VERSION={VERSION} → v3 저장 건너뜀')

ℹ️  VERSION=v4 → v3 저장 건너뜀


---
## 💾 셀 14. v4 저장

**v4 포함 피처: v1 전체 + 네이버 트렌드 피처 (5개)**
- 개봉 후 데이터인 week 관련 데이터(v2) 제외
- 즉 기존 v3데이터에서 | `weekend_ratio` | 주말(토+일) 관람 비율 |
- | `weekday_0_ratio` ~ `weekday_4_ratio` | 월~금 요일별 관람 비율 (5개) | 데이터 제거


In [ ]:
if VERSION == 'v4':
    _save_cols   = ID_COLS + V1_FEATURE_COLS + TREND_COLS + TARGET_COLS
    _save_cols   = [c for c in _save_cols if c in df.columns]
    df_out = df[_save_cols].copy()
    df_out['total_audience'] = df_out['total_audience'].astype(int)
    df_out['hit_class']      = df_out['hit_class'].astype(int)
    df_out.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
    print(f'✅ v4 저장 완료: {OUTPUT_PATH}')
    print(f'   shape: {df_out.shape[0]:,}편 × {df_out.shape[1]}컬럼')
    print(f'   v2 대비 추가 컬럼: {TREND_COLS}')
else:
    print(f'ℹ️  VERSION={VERSION} → v4 저장 건너뜀')

✅ v5 저장 완료: data/processed/feature_table_v4.csv
   shape: 3,869편 × 32컬럼
   v2 대비 추가 컬럼: ['trend_pre7_avg', 'trend_pre7_max', 'trend_growth_rate', 'relative_search_share', 'has_trend_data']


In [38]:
df4 = pd.read_csv('data/processed/feature_table_v4.csv')
df4.info()

<class 'pandas.DataFrame'>
RangeIndex: 3869 entries, 0 to 3868
Data columns (total 32 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   movie_id                 3869 non-null   str    
 1   title                    3869 non-null   str    
 2   runtime                  3869 non-null   float64
 3   rating_encoded           3869 non-null   float64
 4   is_korean                3869 non-null   float64
 5   genre                    3869 non-null   str    
 6   open_date                3869 non-null   str    
 7   open_month               3869 non-null   float64
 8   open_day_of_week         3869 non-null   float64
 9   is_summer                3869 non-null   int64  
 10  is_winter                3869 non-null   int64  
 11  is_holiday_release       3869 non-null   int64  
 12  holiday_nearby_count     3869 non-null   int64  
 13  director_avg_audi        3869 non-null   float64
 14  director_movie_count     3869 non-n